# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Schema: {croissant_url}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list record sets, their `@id`, included fields, and columns. We will reference all entities by their `@id`.

In [ ]:
# List all available record sets and their fields by `@id`

record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"Record Set: name='{rs.name}', @id='{rs.id_}'")
        print(f"  Description: {rs.description}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - name='{field.name}', @id='{field.id_}' (type: {field.data_type})")
        print("  Columns:")
        for col in rs.columns:
            print(f"    - name='{col.name}', @id='{col.id_}' (source: {col.source})")
        print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract all records in each record set into separate DataFrames (referenced by their `@id`).

In [ ]:
# Gather all record set ids
record_set_ids = [rs.id_ for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id='{record_set_id}' ...")
    # Get records as list of dicts
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")

# As an example, list columns and head of first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Example dataframe for @id='{first_rs}':")
    print(dataframes[first_rs].head())
else:
    print("No record sets with data were found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

**Note:** Replace the placeholders for `numeric_field_id` and `group_field_id` with the actual `@id` of a field or column from your record set. See the cell above for options.

In [ ]:
# Set these to the `@id` values of a numeric field and a group field from a record set
# You can refer to the columns printed above to pick the appropriate column name.
record_set_id = None
numeric_field_id = None   # For example: '@id' of 'cr:field:Abundance'
group_field_id = None     # For example: '@id' of 'cr:field:Sample'

# Suggest a record set if available
if record_set_ids:
    record_set_id = record_set_ids[0]
    display_cols = dataframes[record_set_id].columns.tolist()
    # Try to guess likely numeric / groupable fields
    possible_numerics = [c for c in display_cols if any(x in c.lower() for x in ['abundance', 'count', 'mw', 'mass', 'coverage', 'peptide'])]
    if possible_numerics:
        numeric_field_id = possible_numerics[0]
    possible_group = [c for c in display_cols if any(x in c.lower() for x in ['sample', 'type', 'category', 'label']) and c != numeric_field_id]
    if possible_group:
        group_field_id = possible_group[0]

if record_set_id:
    df = dataframes[record_set_id]

    if numeric_field_id and numeric_field_id in df.columns:
        # Remove non-numeric values and cast
        numeric_df = df.copy()
        numeric_df = numeric_df[pd.to_numeric(numeric_df[numeric_field_id], errors='coerce').notnull()]
        numeric_df[numeric_field_id] = pd.to_numeric(numeric_df[numeric_field_id], errors='coerce')
        threshold = numeric_df[numeric_field_id].mean()
        filtered_df = numeric_df[numeric_df[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > mean (≈{threshold:.2f}): {len(filtered_df)} records")
        print(filtered_df.head())
        
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group field if present and numeric
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by '{group_field_id}':")
            print(grouped_df.head())
    else:
        print("No numeric field identified for EDA. Please set 'numeric_field_id' to a column name containing numeric values.")
else:
    print("No record set found for EDA. Please check available record sets and rerun the above cells.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
# Example: visualize the distribution of the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id and numeric_field_id in dataframes[record_set_id].columns:
    df = dataframes[record_set_id].copy()
    df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in {record_set_id}")
    plt.show()

    # If grouping field is present, show boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id} in {record_set_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Unable to plot: Make sure 'numeric_field_id' is set and available in the chosen record set.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset schema defines its structure and metadata using Croissant.
- Record sets, fields, and columns can be programmatically explored and are referenced by `@id`.
- Interactive EDA and visualizations allow for initial pattern discovery and preparation for subsequent ML workflows.

Continue with more domain-specific analysis, model development, or advanced data wrangling as needed. For more, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/).